<a href="https://colab.research.google.com/github/Muaz-Hossain/vehicle_tracking/blob/main/vehicle_tracking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ultralytics deep-sort-realtime opencv-python


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.7/828.7 kB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 26.3 MB/s eta 0:00:00


In [2]:
import cv2
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
model = YOLO("yolov8n.pt", verbose=False)

In [4]:
from google.colab import files
uploaded = files.upload()
video_path = list(uploaded.keys())[0]
print(f"Uploaded video: {video_path}")

Saving Record_2026-05-18-15-22-28_f2cb81fb7cf38af7978f186f2a61634a.mp4 to Record_2026-05-18-15-22-28_f2cb81fb7cf38af7978f186f2a61634a.mp4
Uploaded video: Record_2026-05-18-15-22-28_f2cb81fb7cf38af7978f186f2a61634a.mp4


In [5]:
cap = cv2.VideoCapture(video_path)

In [6]:

from google.colab.patches import cv2_imshow

height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
fps = cap.get(cv2.CAP_PROP_FPS)

output_path = "tracking.mp4"
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out_video = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

tracker = DeepSort(max_age=50, n_init=3, nms_max_overlap=0.5)

line_y = height//2
total_count = 0
frame_skip = 3
frame_count = 0
cross_ids = set()
vehicles_classes = [2, 3, 5, 7]

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_count += 1
    if frame_count % frame_skip != 0:
        continue
    #run yolo
    results = model.track(frame, persist=True,verbose=False,classes = vehicles_classes)
    detections = []

    for box in results[0].boxes:
      cls_id = int(box.cls[0])

      if cls_id in vehicles_classes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        conf = float(box.conf[0])
        detections.append(([x1, y1, x2, y2], conf, cls_id))
    #update tracker
    tracks= tracker.update_tracks(detections, frame=frame)
    for track in tracks:
      if not track.is_confirmed():
        continue
      track_id = track.track_id
      ltrb = track.to_ltrb()
      x1, y1, x2, y2 = map(int, ltrb)
      #draw boundig box
      cx = (x1+x2)//2
      cy = (y1+y2)//2

      label = f"{model.names[cls_id]} {conf:.2f}"
      cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0),2)
      cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

      if cy> line_y and track_id not in cross_ids:
        cross_ids.add(track_id)
        total_count += 1
        print(f"vehicle {track_id} total {total_count}")
    #run deepsort
    cv2.line(frame, (0, line_y), (width, line_y), (0, 0, 255), 2)
    cv2.putText(frame, f"Total: {total_count}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
    out_video.write(frame)
    #show frame count
    if frame_count % 50 == 0:

      continue

cap.release()
out_video.release()
cv2.destroyAllWindows()






requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 227ms
Prepared 1 package in 32ms
Installed 1 package in 3ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 2.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

vehicle 1 total 1
vehicle 2 total 2
vehicle 6 total 3
vehicle 9 total 4
vehicle 11 total 5
vehicle 12 total 6
vehicle 13 total 7
vehicle 14 total 8
vehicle 3 total 9
vehicle 18 total 10
vehicle 23 total 11
vehicle 24 total 12
vehicle 26 total 13
vehicle 33 total 14
vehicle 35 total 15
vehicle 4 total 16
vehicle 43 total 17
vehicle 48 total 18
vehicle 50 total 19
vehicle 47 total 20
vehicle 55 total 21
vehicle 67 total 22
vehicle 65 total 23
vehicle 32 total 24
vehicle 16 total 25
vehicle 68 total 26
vehicle 71 total 27
vehicle 59 total 28
